In [ ]:
!pip install pymupdf
!pip install sentence-transformers
!pip install joblib
!pip install numpy

In [ ]:
import os
import re
import joblib
import numpy as np
import pandas as pd
import pymupdf

from sentence_transformers import SentenceTransformer

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
PDF_PATH= '/content/drive/MyDrive/Colab Notebooks/ICC_handbook.pdf'

In [ ]:
def extract_pdf_pages(PDF_PATH):
    doc = pymupdf.open(PDF_PATH)

    pages = []

    for page_number, page in enumerate(doc, start=1):
        text = page.get_text("text")

        if text and text.strip():
            pages.append({
                "page": page_number,
                "text": text.strip()
            })

    doc.close()

    return pages


pages = extract_pdf_pages(PDF_PATH)

print(f"Total pages with text: {len(pages)}")

Total pages with text: 664


In [ ]:
for page in pages[:3]:
    print("=" * 80)
    print("PAGE:", page["page"])
    print(page["text"][:2000])

PAGE: 1
INTERNATIONAL LAW HANDBOOK
COLLECTION OF INSTRUMENTS
BOOK ONE
PAGE: 2
The photograph on the cover is of a stained 
glass window in the United Nations 
Headquarters building in New York. The 
staff of the United Nations and Marc Chagall 
donated the stained glass panel designed 
by the French artist as a memorial to Dag 
Hammarskjöld and 15 others who died in a 
plane crash while on a peace mission in the 
Congo in 1961. Dag Hammarskjöld served as 
the second Secretary-General of the United 
Nations from 10 April 1953 until his death 
on 18 September 1961. He introduced the 
concept of peacekeeping and was awarded 
the Nobel Peace Prize. He also defined the 
role of an international civil servant based on his personal devotion to the 
Charter of the United Nations and to public service.
In the panel Chagall sought to express the simplicity and beauty of the 
ideals of peace and brotherhood for which the United Nations was 
founded. Symbols of peace and love can be found througho

In [ ]:
def clean_text(text):
    # Replace multiple spaces/tabs with one space
    text = re.sub(r"[ \t]+", " ", text)

    # Replace excessive newlines
    text = re.sub(r"\n+", "\n", text)

    # Remove spaces at beginning/end
    text = text.strip()

    return text


for page in pages:
    page["text"] = clean_text(page["text"])

print(pages[0]["text"][:2000])

INTERNATIONAL LAW HANDBOOK
COLLECTION OF INSTRUMENTS
BOOK ONE


In [ ]:
def chunk_text(text, chunk_size=250, overlap=50):
    words = text.split()

    chunks = []

    start = 0

    while start < len(words):
        end = start + chunk_size

        chunk = " ".join(words[start:end])

        if chunk.strip():
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [ ]:
all_chunks = []

for page_data in pages:
    page_number = page_data["page"]
    text = page_data["text"]

    page_chunks = chunk_text(
        text,
        chunk_size=250,
        overlap=50
    )

    for chunk_id, chunk in enumerate(page_chunks):
        all_chunks.append({
            "chunk_id": len(all_chunks),
            "page": page_number,
            "chunk_on_page": chunk_id,
            "text": chunk
        })


print("Total chunks:", len(all_chunks))

Total chunks: 2022


In [ ]:
for chunk in all_chunks[:5]:
    print("=" * 80)
    print("Chunk ID:", chunk["chunk_id"])
    print("Page:", chunk["page"])
    print("Text:")
    print(chunk["text"][:1000])

Chunk ID: 0
Page: 1
Text:
INTERNATIONAL LAW HANDBOOK COLLECTION OF INSTRUMENTS BOOK ONE
Chunk ID: 1
Page: 2
Text:
The photograph on the cover is of a stained glass window in the United Nations Headquarters building in New York. The staff of the United Nations and Marc Chagall donated the stained glass panel designed by the French artist as a memorial to Dag Hammarskjöld and 15 others who died in a plane crash while on a peace mission in the Congo in 1961. Dag Hammarskjöld served as the second Secretary-General of the United Nations from 10 April 1953 until his death on 18 September 1961. He introduced the concept of peacekeeping and was awarded the Nobel Peace Prize. He also defined the role of an international civil servant based on his personal devotion to the Charter of the United Nations and to public service. In the panel Chagall sought to express the simplicity and beauty of the ideals of peace and brotherhood for which the United Nations was founded. Symbols of peace and love ca

In [ ]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

model = SentenceTransformer(MODEL_NAME)

print("Model loaded:", MODEL_NAME)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded: sentence-transformers/all-MiniLM-L6-v2


In [ ]:
texts = [chunk["text"] for chunk in all_chunks]

print("Number of texts:", len(texts))

Number of texts: 2022


In [ ]:
embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings = np.asarray(embeddings, dtype=np.float32)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/64 [00:00<?, ?it/s]

Embedding shape: (2022, 384)


In [ ]:
for i, chunk in enumerate(all_chunks):
    chunk["embedding"] = embeddings[i]

print(all_chunks[0].keys())

dict_keys(['chunk_id', 'page', 'chunk_on_page', 'text', 'embedding'])


In [ ]:
texts = [chunk["text"] for chunk in all_chunks]

metadata = [
    {
        "chunk_id": chunk["chunk_id"],
        "page": chunk["page"],
        "chunk_on_page": chunk["chunk_on_page"]
    }
    for chunk in all_chunks
]

data = {
    "embeddings": embeddings,
    "texts": texts,
    "metadata": metadata,
    "model_name": MODEL_NAME,
    "pdf_name": os.path.basename(PDF_PATH),
    "chunk_size": 250,
    "chunk_overlap": 50
}

print("Data prepared.")
print("Embeddings:", data["embeddings"].shape)
print("Texts:", len(data["texts"]))
print("Metadata:", len(data["metadata"]))

Data prepared.
Embeddings: (2022, 384)
Texts: 2022
Metadata: 2022


In [ ]:
OUTPUT_FILE = "international_law_embeddings.joblib"

joblib.dump(
    data,
    OUTPUT_FILE,
    compress=3
)

print(f"Saved successfully: {OUTPUT_FILE}")

Saved successfully: international_law_embeddings.joblib
